In [1]:
import asyncio
import os

import requests
from dotenv import load_dotenv

from agents import Agent, ModelSettings, Runner, function_tool, trace

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"

In [2]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print("user set:", bool(pushover_user))
print("token set:", bool(pushover_token))

user set: True
token set: True


In [3]:
def push(message):
    print(f"Push: {message[:100]}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message[:1000]}
    response = requests.post(pushover_url, data=payload, timeout=10)
    print("Status:", response.status_code)

In [5]:
def send_message(subject, body):
    push(f"Subject: {subject}\n\n{body}")

In [6]:
send_message("Test subject", "Communications are a go!")

Push: Subject: Test subject

Communications are a go!
Status: 200


## three sales agents with different writing styles

In [7]:
intro = """
You are a sales agent working for CloudWatchly,
a company that provides a SaaS tool for monitoring cloud costs and flagging waste, powered by AI.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [9]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)

In [10]:
result = await Runner.run(sales_agent1, "Write a cold sales email")
print(result.final_output)

Subject: Reducing cloud waste without adding workload

Hi [First Name],

I’m reaching out from CloudWatchly. We help finance and engineering teams identify cloud cost waste, surface anomalies early, and prioritize the savings that matter most.

What sets us apart is that CloudWatchly uses AI to continuously monitor usage patterns and flag waste automatically, so your team spends less time hunting for inefficiencies and more time acting on them.

If cloud spend is under pressure, I’d welcome the chance to show you how we help companies regain control without adding operational burden.

Would you be open to a brief conversation next week?

Best,  
[Your Name]  
CloudWatchly


## run all three agents at the same time

In [11]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Reducing cloud waste without adding overhead

Hi [First Name],

I’m reaching out from CloudWatchly. We help teams monitor cloud spend, identify waste, and surface savings opportunities using AI—without adding more manual work to engineering or finance.

Most organizations know cloud costs are rising, but the harder problem is finding where the waste is hiding. CloudWatchly continuously flags idle resources, overprovisioning, and unusual spend patterns so your team can act quickly and with confidence.

If reducing cloud waste is a priority this quarter, I’d welcome a brief conversation to see whether we could be useful.

Best,  
[Your Name]  
CloudWatchly


Subject: Your cloud bill called. It has questions.

Hi [First Name],

I’ll keep this short, unlike most cloud invoices.

CloudWatchly helps teams spot cloud waste before it turns into a “how is this bill even possible?” moment. We monitor spend, flag inefficiencies, and use AI to surface the stuff that’s quietly burning budg

In [12]:
import time

start = time.time()
await Runner.run(sales_agent1, message)
await Runner.run(sales_agent2, message)
await Runner.run(sales_agent3, message)
print("One after another:", round(time.time() - start, 1), "seconds")

One after another: 7.7 seconds


In [13]:
start = time.time()
await asyncio.gather(
    Runner.run(sales_agent1, message),
    Runner.run(sales_agent2, message),
    Runner.run(sales_agent3, message),
)
print("Together:", round(time.time() - start, 1), "seconds")

Together: 3.7 seconds


In [14]:
## a picker agent chooses the best email (orchestration by code)
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales Picker", instructions=decision, model=MODEL_NAME)

In [15]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Cut cloud waste without adding overhead

Hi {{FirstName}},

Cloud spend often creeps up quietly — idle resources, overprovisioned instances, forgotten services.

CloudWatchly helps teams spot waste fast with AI-driven alerts and cost monitoring, so you can reduce spend without manual digging.

Worth a quick 15-minute chat to see if there’s an opportunity in your environment?

Best,  
{{YourName}}


In [17]:
## give an agent a tool to send the email

@function_tool
def send_email_tool(subject: str, body: str) -> str:
    """
    Send the chosen sales email to the user as a notification

    Args:
        subject: The subject line of the email
        body: The body of the email as plain text
    """
    send_message(subject, body)
    return "Email sent successfully"

In [18]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject line of the email',
   'title': 'Subject',
   'type': 'string'},
  'body': {'description': 'The body of the email as plain text',
   'title': 'Body',
   'type': 'string'}},
 'required': ['subject', 'body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [19]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")

sales_sender = Agent(
    name="Sales Sender",
    instructions=decision,
    model=MODEL_NAME,
    tools=[send_email_tool],
    model_settings=require_tool,
)

In [20]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

Push: Subject: Reducing cloud waste without adding operational overhead

Hi [First Name],

I’m reaching ou
Status: 200
Final response:
Sent the email with the first option:
**“Reducing cloud waste without adding operational overhead”**


## agents as tools, with a Sales Manager (orchestration by an LLM)

In [22]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

manager_tools = [tool1, tool2, tool3, send_email_tool]
manager_tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000012CBAAFF890>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input

In [23]:
manager_instructions = """
You are a Sales Manager at CloudWatchly. Your goal is to find the single best cold sales email using the sales_email_writer tools.
"""

manager_task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.

3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=manager_tools,
    model=MODEL_NAME,
)

In [24]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, manager_task)

print(result.final_output)

Push: Subject: Cut cloud waste without adding headcount

Hi [First Name],

Cloud spend keeps creeping up b
Status: 200
Sent.


## handoffs, where one agent passes control to another

In [25]:
writer_tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

In [27]:
handoff_instructions = """
You are a Sales Manager at CloudWatchly. You get your sales team to draft emails, then hand off to the sales sender.
"""

handoff_task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Handoff to the sales sender to choose and send the best email.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=handoff_instructions,
    tools=writer_tools,
    handoffs=handoffs,
    model=MODEL_NAME,
)

In [28]:
with trace("Sales manager with handoff"):
    result = await Runner.run(sales_manager, handoff_task)

print(result.final_output)

Push: Subject: Cut cloud waste without adding work

Hi [First Name],

Cloud costs keep creeping up, and mo
Status: 200
Sent the best-performing email.


In [29]:
critic = Agent(
    name="Email Critic",
    instructions="""
You review a cold sales email. Give two or three short, specific suggestions to make it more likely to get a reply.
Do not rewrite the email.
""",
    model=MODEL_NAME,
)

reviser = Agent(
    name="Email Reviser",
    instructions="""
You improve a cold sales email using the feedback you are given.
Reply with the improved email only, with no explanation.
""",
    model=MODEL_NAME,
)

In [30]:
async def refine(email, rounds=2):
    for round_number in range(rounds):
        critique = await Runner.run(critic, f"Email:\n\n{email}")

        revised = await Runner.run(
            reviser,
            f"Email:\n\n{email}\n\nFeedback:\n\n{critique.final_output}",
        )
        email = revised.final_output
        print(f"Round {round_number + 1} done")

    return email

In [31]:
message = "Write a cold sales email"

with trace("Sales workflow with refinement"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]
    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)
    final_email = await refine(best.final_output, rounds=2)

send_message("Refined cold email", final_email)
print(final_email)

Round 1 done
Round 2 done
Push: Subject: Refined cold email

Subject: Cut cloud waste without adding headcount

Hi [First Name],

Co
Status: 200
Subject: Cut cloud waste without adding headcount

Hi [First Name],

Congrats on [recent hiring / product launch / cloud migration] at [Company] — that kind of growth usually means cloud costs can climb fast too.

I’m reaching out from CloudWatchly. We help teams spot cloud waste early — idle resources, overprovisioned spend, and unusual spikes — so Finance and Engineering can act before it compounds.

For example, a similar [industry/company type] team used CloudWatchly to identify $84K in avoidable monthly spend in their first audit.

If helpful, I can put together a quick cloud cost-savings snapshot for [Company] and send it over — no prep needed on your side.

Best,  
[Your Name]  
CloudWatchly
